# Refactored Weekly Meal Planner

Path 2: refactor the supplied starter notebook. Run setup, definitions and local tests first; live API/email checks require credentials.

In [1]:
%pip install -q langchain langchain-openai pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, time, smtplib, logging
from pathlib import Path
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import List
from enum import Enum
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from openai import AuthenticationError, RateLimitError, APITimeoutError, APIConnectionError, APIStatusError
load_dotenv()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TEMPERATURE = 0.7
EMAIL_SENDER = os.getenv("EMAIL_SENDER", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")
INGREDIENTS_PER_RECIPE = 5
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("meal_planner")
def require_openai_key():
    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError("require_openai_key | ValueError | OPENAI_API_KEY missing from environment | Set it in .env")
def normalize_optional(value, default):
    return value.strip() if isinstance(value, str) and value.strip() else default
def format_ingredients(recipe):
    return ", ".join(f"{i['quantity']} {i['name']}" for i in recipe["ingredients"])
def failure(operation, exc, suggestion):
    logger.error("%s | %s | %s", operation, type(exc).__name__, str(exc))
    return {"success": False, "meal_plan": None, "generation_time": None,
            "error": f"{operation} | {type(exc).__name__}: {exc} | Suggestion: {suggestion}"}
def parse_json_response(content):
    if not isinstance(content, str):
        raise TypeError("Model response content must be text")
    text = content.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if len(lines) < 3 or lines[-1].strip() != "```":
            raise ValueError("Incomplete fenced JSON response")
        text = "\n".join(lines[1:-1])
    return json.loads(text)
def load_test_cases(path):
    with open(path, encoding="utf-8") as handle:
        cases = json.load(handle)
    if not isinstance(cases, list) or not all(isinstance(c, dict) for c in cases):
        raise ValueError("Test case file must contain a list of objects")
    return cases


In [3]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != 7:
            raise ValueError(f"Meal plan must have exactly 7 recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All 7 themes must be unique")
        return v

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def get_llm():
    """Get the configured LLM."""
    require_openai_key()
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])




In [5]:
class MealPlanGenerator:
    def __init__(self, llm=None):
        self.llm = llm  # Allows offline fake model in tests.
        self.recipe_parser = JsonOutputParser(pydantic_object=MealPlan)

    def request_model(self, dietary_preference, theme_requests, additional_preferences):
        model = self.llm if self.llm is not None else get_llm()
        chain = MEAL_PLAN_PROMPT | model
        return chain.invoke({
            "dietary_preference": dietary_preference,
            "theme_requests": normalize_optional(theme_requests, "random diverse cuisines"),
            "additional_preferences": normalize_optional(additional_preferences, "none"),
            "format_instructions": self.recipe_parser.get_format_instructions()
        })

    def validate_response(self, response):
        data = parse_json_response(response.content)
        return MealPlan.model_validate(data).model_dump()

    def generate(self, dietary_preference, theme_requests="random", additional_preferences="none"):
        start = time.monotonic()
        try:
            response = self.request_model(dietary_preference, theme_requests, additional_preferences)
            plan = self.validate_response(response)
            return {"success": True, "meal_plan": plan,
                    "generation_time": round(time.monotonic() - start, 2), "error": None}
        except (AuthenticationError, RateLimitError, APITimeoutError, APIConnectionError, APIStatusError) as exc:
            return failure("MealPlanGenerator.generate/request_model", exc,
                           "Check credentials, connectivity, service status or retry policy as appropriate")
        except json.JSONDecodeError as exc:
            return failure("MealPlanGenerator.generate/parse_json_response", exc,
                           f"Inspect model JSON near line {exc.lineno}, column {exc.colno}")
        except ValidationError as exc:
            return failure("MealPlanGenerator.generate/validate_response", exc,
                           "Inspect the Pydantic field errors and correct the model output")
        except (ValueError, TypeError, AttributeError) as exc:
            return failure("MealPlanGenerator.generate", exc,
                           "Check configuration and model response format")
        except Exception as exc:
            logger.exception("Unexpected generation failure")
            return failure("MealPlanGenerator.generate", exc, "Inspect diagnostic logs")

generator = MealPlanGenerator()


In [6]:
def format_meal_plan_email(meal_plan):
    lines = ["=" * 40, f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})",
             "=" * 40, "", "SHOPPING LIST", "-" * 20]
    lines.extend(f"- {item['total_quantity']} {item['name']}" for item in meal_plan["shopping_list"])
    lines.extend(["", "=" * 40, "RECIPES", "=" * 40])
    for recipe in meal_plan["recipes"]:
        lines.extend([f"\n{recipe['day'].upper()} - {recipe['theme']}",
                      recipe["dish_name"], f"Ingredients: {format_ingredients(recipe)}",
                      f"Instructions: {recipe['instructions']}"])
    return "\n".join(lines)

def build_email_message(recipient, plan):
    message = MIMEMultipart()
    message["From"] = EMAIL_SENDER
    message["To"] = recipient
    message["Subject"] = "Your Weekly Meal Plan"
    message.attach(MIMEText(format_meal_plan_email(plan), "plain"))
    return message

def send_meal_plan_email(recipient_email, meal_plan):
    try:
        if not EMAIL_SENDER or not EMAIL_PASSWORD:
            raise ValueError("EMAIL_SENDER or EMAIL_PASSWORD missing from environment")
        if not recipient_email or "@" not in recipient_email:
            raise ValueError("Invalid recipient email")
        msg = build_email_message(recipient_email, meal_plan)
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.send_message(msg)
        return {"success": True, "message": f"Email sent to {recipient_email}"}
    except (ValueError, smtplib.SMTPException, OSError) as exc:
        logger.error("send_meal_plan_email | %s | %s", type(exc).__name__, exc)
        return {"success": False, "message":
                f"send_meal_plan_email | {type(exc).__name__}: {exc} | Check email configuration or SMTP connectivity"}


In [7]:
def run_basic_validation(result):
    if not result.get("success"):
        return {"passed": False, "errors": [result.get("error")]}
    try:
        MealPlan.model_validate(result["meal_plan"])
        return {"passed": True, "errors": []}
    except ValidationError as exc:
        return {"passed": False, "errors": [str(exc)]}

def format_shopping_list(plan):
    return "\n".join(["## Shopping List\n"] +
                     [f"- {item['total_quantity']} **{item['name']}**" for item in plan["shopping_list"]])

def format_single_recipe(recipe):
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{format_ingredients(recipe)}\n"

def render_plan(plan, duration):
    output = f"## Weekly {plan['dietary_preference'].title()} Meal Plan\n\n"
    output += format_shopping_list(plan) + "\n\n---\n\n## Recipes\n\n"
    for recipe in plan["recipes"]:
        output += format_single_recipe(recipe) + "\n"
    output += f"---\n*Generated in {duration}s | All checks passed*"
    return output


In [8]:
import gradio as gr
def generate_plan_ui(dietary_pref, theme_requests, additional_prefs):
    result = generator.generate(dietary_pref, theme_requests, additional_prefs)
    if not result["success"]:
        return f"Error: {result['error']}", None
    return render_plan(result["meal_plan"], result["generation_time"]), result["meal_plan"]

def send_email_ui(email, saved_plan):
    if saved_plan is None:
        return "Generate a meal plan before emailing it."
    return send_meal_plan_email(email, saved_plan)["message"]

with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")
    saved_plan = gr.State(value=None)
    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(choices=["omnivore","vegetarian","vegan","keto"],
                                           value="omnivore", label="Diet")
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...")
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts...")
            generate_btn = gr.Button("Generate", variant="primary")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)
        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")
    generate_btn.click(generate_plan_ui,
                       [dietary_dropdown, theme_input, additional_input],
                       [output_display, saved_plan])
    send_btn.click(send_email_ui, [email_input, saved_plan], [email_status])

# To start the UI, run app.launch() manually after tests.


c:\Users\sabri\miniconda3\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"


In [10]:
# OFFLINE CHECKS: do not call OpenAI or send email.
from types import SimpleNamespace
sample = {
 "dietary_preference": "vegetarian",
 "recipes": [
    {"day": day, "theme": theme, "dish_name": f"{theme} dinner",
     "ingredients": [{"name": f"item{n}", "quantity": "1 cup"} for n in range(5)],
     "instructions": "Cook and serve."}
    for day, theme in zip(DAYS, ["Italian","Mexican","Japanese","Indian","Thai","Greek","French"])
 ],
 "shopping_list": [{"name": "item0", "total_quantity": "7 cups",
                    "used_in": ["Italian dinner"]}]
}
assert MealPlan.model_validate(sample)
assert len(load_test_cases("test_cases.json")) == 4
assert normalize_optional(None, "none") == "none"
assert "1 cup item0" in format_ingredients(sample["recipes"][0])
assert parse_json_response("```json\n{\"ok\": true}\n```") == {"ok": True}
try:
    parse_json_response('{"recipes": [}')
except json.JSONDecodeError as exc:
    print("EXPECTED malformed JSON:", type(exc).__name__, "line", exc.lineno, "column", exc.colno)
else:
    raise AssertionError("Malformed JSON was accepted")
bad = json.loads(json.dumps(sample))
bad["recipes"] = bad["recipes"][:-1]
try:
    MealPlan.model_validate(bad)
except ValidationError as exc:
    print("EXPECTED schema failure:", type(exc).__name__, exc.errors()[0]["loc"])
else:
    raise AssertionError("Six recipes were accepted")
try:
    load_test_cases("missing_test_cases.json")
except FileNotFoundError as exc:
    print("EXPECTED missing file:", type(exc).__name__, exc.filename)
else:
    raise AssertionError("Missing file was accepted")
try:
    load_test_cases("malformed.json")
except json.JSONDecodeError as exc:
    print("EXPECTED malformed file:", type(exc).__name__, exc.lineno, exc.colno)
else:
    raise AssertionError("Malformed file was accepted")
from langchain_core.runnables import RunnableLambda
offline_generator = MealPlanGenerator(llm=RunnableLambda(lambda inputs: SimpleNamespace(content=json.dumps(sample))))
result = offline_generator.generate("vegetarian")
assert result["success"] and len(result["meal_plan"]["recipes"]) == 7
assert "Italian dinner" in render_plan(result["meal_plan"], result["generation_time"])
assert send_email_ui("test@example.com", None) == "Generate a meal plan before emailing it."
print("PASS: offline valid plan, parsing, validation, file errors and UI state checks")


EXPECTED malformed JSON: JSONDecodeError line 1 column 14
EXPECTED schema failure: ValidationError ('recipes',)
EXPECTED missing file: FileNotFoundError missing_test_cases.json
EXPECTED malformed file: JSONDecodeError 1 14
PASS: offline valid plan, parsing, validation, file errors and UI state checks


In [ ]:
app.launch()


INFO HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


INFO HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [12]:
import httpx

test_generator = MealPlanGenerator()

def simulate_timeout(*args, **kwargs):
    raise APITimeoutError(
        request=httpx.Request("POST", "https://api.openai.com/v1/chat/completions")
    )

test_generator.request_model = simulate_timeout

result = test_generator.generate("vegetarian")

print("Success:", result["success"])
print("Error:", result["error"])

assert result["success"] is False
assert "APITimeoutError" in result["error"]

print("PASS: API timeout handled correctly")

ERROR MealPlanGenerator.generate/request_model | APITimeoutError | Request timed out.


Success: False
Error: MealPlanGenerator.generate/request_model | APITimeoutError: Request timed out. | Suggestion: Check credentials, connectivity, service status or retry policy as appropriate
PASS: API timeout handled correctly
